# DSA 504 — Class 23
## Reliable Code: Debugging and Testing

**Date:** Monday, Nov 23
**Today:** HW6 is due before class.

---

Every class this semester has focused on *doing* something with data. Today is different: it's about trusting the code you've already written. Real data work is full of small mistakes — a typo, a wrong assumption about what's in a column, a function that silently gives the wrong answer instead of crashing. Today covers how to find those mistakes efficiently, and how to catch them before they become a problem.

### Learning goals
By the end of this class, you will be able to:
- Read a Python traceback and find the actual error, instead of being overwhelmed by it
- Recognize common error types and what each one usually means
- Use `assert` statements to check an assumption your code depends on
- Write a simple test function that checks whether code behaves correctly
- Debug code that gives a *wrong* answer without crashing — often harder than fixing a crash


In [1]:
import pandas as pd
import numpy as np

sales = pd.read_csv("retail_sales_clean.csv")
sales.head()


,date,store,units_sold,revenue,category,is_end_of_month
0,2025-04-22,Rome,25.0,402.50,toys,False
1,2025-04-12,Syracuse,38.0,3154.00,electronics,False
2,2025-12-21,Utica,42.0,1299.06,home goods,False
3,2025-02-11,Syracuse,60.0,5114.40,electronics,False
4,2025-04-10,Syracuse,10.0,147.10,toys,False


## 1. Reading a Traceback

A **traceback** is the error message Python prints when something goes wrong. It looks intimidating, but it's actually structured and readable once you know where to look.

**The two most important things in any traceback:**
1. **The last line** — this names the actual error type and gives a short message. Start here, not at the top.
2. **The line number** right above it — this tells you exactly where in your code the problem occurred.

Everything in between is the "call stack" — the chain of function calls that led to the error. For now, the bottom line and that final line number are 90% of what you need.


In [2]:
# Let's trigger a real error and read its traceback
d = {"store": "Utica", "revenue": 4500}
print(d["catgory"])   # typo: should be "category" -- but this key doesn't exist at all


KeyError: 'catgory'

**Reading this traceback:** the last line says `KeyError: 'catgory'`. That tells you immediately: you tried to access a dictionary (or DataFrame column) key that doesn't exist, and the key you tried was `'catgory'` — which, reading it again, is simply misspelled. No need to read every line of the traceback above it; the last line already told you exactly what's wrong.


## 2. Common Error Types and What They Usually Mean

Each error type is a category of mistake. Recognizing the type narrows down what to look for immediately.


In [3]:
# KeyError -- a dictionary or DataFrame column/index key doesn't exist
try:
    sales["revnue"]   # typo
except KeyError as e:
    print("KeyError:", e)


KeyError: 'revnue'


In [4]:
# TypeError -- an operation doesn't make sense for the types involved
try:
    result = "Total: " + 500   # can't add a string and a number directly
except TypeError as e:
    print("TypeError:", e)


TypeError: can only concatenate str (not "int") to str


In [5]:
# ValueError -- the type is right, but the actual value doesn't make sense for this operation
try:
    int("not a number")
except ValueError as e:
    print("ValueError:", e)


ValueError: invalid literal for int() with base 10: 'not a number'


In [6]:
# IndexError -- trying to access a position in a list that doesn't exist
try:
    stores = ["Utica", "Albany", "Rome"]
    stores[10]
except IndexError as e:
    print("IndexError:", e)


IndexError: list index out of range


In [7]:
# FileNotFoundError -- exactly what it says
try:
    pd.read_csv("this_file_does_not_exist.csv")
except FileNotFoundError as e:
    print("FileNotFoundError:", e)


FileNotFoundError: [Errno 2] No such file or directory: 'this_file_does_not_exist.csv'


In [8]:
# NameError -- using a variable that was never defined (often a typo, or running cells out of order)
try:
    print(toatl_revenue)
except NameError as e:
    print("NameError:", e)


NameError: name 'toatl_revenue' is not defined


In [9]:
# AttributeError -- calling a method that doesn't exist on this type of object
try:
    sales.sumarize()   # typo for .describe(), or just a method that doesn't exist
except AttributeError as e:
    print("AttributeError:", e)


AttributeError: 'DataFrame' object has no attribute 'sumarize'


**A quick mental map:**
- `KeyError` / `IndexError` → you're looking for something that isn't there, by name or by position
- `TypeError` → you're combining or using things in a way their types don't support
- `ValueError` → the type is fine, but this specific value won't work here
- `NameError` → the variable doesn't exist yet — often means a cell above wasn't run, or a typo
- `FileNotFoundError` → exactly what it says — check the filename and the folder you're running from
- `AttributeError` → this object doesn't have that method or property — often a typo, or you're assuming a feature from a different type of object (e.g., trying to use a pandas method on a plain Python list)


## 3. `assert` — Checking an Assumption

An `assert` statement checks that something is `True`, and stops your program immediately with a clear message if it isn't. This turns a silent, hidden assumption into something that fails loudly the moment it's violated — much better than finding out three steps later that something was wrong all along.


In [12]:
assert len(sales) > 0, "Dataset is empty!"
print("Check passed: dataset is not empty")


Check passed: dataset is not empty


In [13]:
assert sales["revenue"].isna().sum() == 0, "There are missing revenue values!"
print("Check passed: no missing revenue values")


Check passed: no missing revenue values


**Seeing an assert actually fail:**


In [ ]:
try:
    assert len(sales) == 100, f"Expected 100 rows, got {len(sales)}"
except AssertionError as e:
    print("AssertionError:", e)


**Where to use asserts in real data work:** right after loading data ("this file should have rows"), right after cleaning ("there should be zero missing values now"), and right before a critical calculation ("this column should never be negative"). An assert is a cheap, one-line way to make a hidden assumption visible and checked automatically, every time the code runs.


## 4. Writing a Simple Test Function

A **test** is a small, separate piece of code whose entire job is to check that another piece of code behaves correctly — usually by calling it with known input and checking the output against a known correct answer.


In [14]:
def clean_revenue(df):
    df = df.copy()
    df["revenue"] = df["revenue"].fillna(df["revenue"].median())
    return df

def test_clean_revenue_removes_missing():
    test_df = pd.DataFrame({"revenue": [10.0, None, 30.0]})
    result = clean_revenue(test_df)
    assert result["revenue"].isna().sum() == 0, "Cleaning should remove all missing values"
    print("test_clean_revenue_removes_missing: PASSED")

test_clean_revenue_removes_missing()


test_clean_revenue_removes_missing: PASSED


**Why bother writing this as a separate function, instead of just checking by eye?** Two reasons: first, you (or someone else) can re-run this exact check in two seconds, any time, instead of re-inspecting the output by hand. Second, it documents *what the function is supposed to do* in a way that's actually verified, not just described in a comment that could silently go stale.

A real project usually has many small test functions like this one — each testing one specific behavior. Tools like `pytest` can automatically find and run every function starting with `test_` in a project, but the core idea is exactly what you just saw: call the function, check the result, use `assert`.


## 5. The Harder Case: Code That Doesn't Crash, But Is Wrong

Every example so far has *crashed* — which is actually the easier case, because Python tells you exactly where and why. The harder, more dangerous bug is code that runs fine and produces a plausible-looking answer that's simply incorrect.


In [15]:
# This function claims to compute AVERAGE revenue per category -- but look closely
def broken_average_revenue(df):
    result = df.groupby("category")["revenue"].sum()   # bug: this is a sum, not an average
    return result

broken_average_revenue(sales)


category
clothing       1261966.80
electronics    2826160.04
groceries       705730.84
home goods      970427.95
toys            315529.51
Name: revenue, dtype: float64

**Nothing crashed. The output even looks reasonable** — a number for every category, nicely formatted. But read the function's own name and intention again: it's supposed to be an *average*, and `.sum()` computes a total, not an average. This kind of bug is genuinely more dangerous than a crash, because nothing alerts you that something is wrong — you have to already know what a correct answer should roughly look like, or test against it, to catch it.


In [16]:
# The fix
def fixed_average_revenue(df):
    result = df.groupby("category")["revenue"].mean()
    return result

fixed_average_revenue(sales)


category
clothing       1542.746699
electronics    3689.503969
groceries       850.278120
home goods     1199.540111
toys            398.899507
Name: revenue, dtype: float64

**How would you have caught this without already knowing the bug was there?** A test with known input and a known correct output — exactly the pattern from Section 4 — would catch this immediately, since `.sum()` and `.mean()` only agree by coincidence when a group has exactly one row.


In [17]:
# A combined condition bug -- using Python's `and` instead of pandas' `&`
def broken_high_value_report(df, store_name):
    result = df[df["store"] == store_name and df["revenue"] > 1000]
    return result

try:
    broken_high_value_report(sales, "Utica")
except ValueError as e:
    print("ValueError:", e)


ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().


**Recognize this one from Class 5?** `and`/`or` are Python's own keywords, meant for combining single `True`/`False` values — they don't know how to handle an entire column of them at once. pandas needs `&`/`|` instead, with each condition wrapped in parentheses. The error message itself is actually a strong hint: "the truth value of a Series is ambiguous" is pandas specifically telling you that you handed it a whole column where it expected one single `True` or `False`.


In [18]:
# The fix
def fixed_high_value_report(df, store_name):
    result = df[(df["store"] == store_name) & (df["revenue"] > 1000)]
    return result

fixed_high_value_report(sales, "Utica").head()


,date,store,units_sold,revenue,category,is_end_of_month
2,2025-12-21,Utica,42.0,1299.06,home goods,False
5,2025-05-07,Utica,39.0,3531.06,electronics,False
15,2025-05-14,Utica,64.0,1578.24,clothing,False
18,2025-03-13,Utica,38.0,1266.16,home goods,False
27,2025-05-20,Utica,50.0,1663.00,home goods,False


---
## Guided Practice

Work through these using `sales`, already loaded above.


### Exercise 1 — Read the traceback
Run the broken code below. Without looking at the solution, answer: what error type is it, and what's actually wrong?
```python
sales["store"].append("Plattsburgh")
```


In [ ]:
# Exercise 1 — run this, then write your answer in the markdown cell below



*Your answer: error type = ___, actual problem = ___*


### Exercise 2 — Write an assert
Write an assert statement that checks every value in `sales["units_sold"]` is greater than or equal to 0 (negative units sold wouldn't make sense).


In [ ]:
# Exercise 2 — your code here



### Exercise 3 — Write a test function
Write a function `add_tax(df)` that adds a new column `revenue_with_tax` (revenue * 1.08). Then write a test function that checks it on a small known example.


In [ ]:
# Exercise 3 — your code here



### Exercise 4 — Find the silent bug
The function below is supposed to return the store with the **highest** total revenue. It doesn't crash, and it does return a store name — but it's wrong. Find and fix the bug.
```python
def highest_revenue_store(df):
    totals = df.groupby("store")["revenue"].sum()
    return totals.sort_values().index[0]   # bug is somewhere in this line
```


In [ ]:
# Exercise 4 — your code here (write both the broken version's wrong answer and your fixed version)



---
## Solutions

Try each exercise yourself first. These are here for after class, or once you're stuck.


In [ ]:
# Solution — Exercise 1
try:
    sales["store"].append("Plattsburgh")
except AttributeError as e:
    print("AttributeError:", e)
# A pandas Series (a column) doesn't have an .append() method the way a Python
# list does -- that's the actual mistake. (Also, even for a list, .append()
# modifies in place and returns None, so this wouldn't do what was likely
# intended even if it ran.)


In [ ]:
# Solution — Exercise 2
assert (sales["units_sold"] >= 0).all(), "Found a negative units_sold value!"
print("Check passed: no negative units_sold values")


In [ ]:
# Solution — Exercise 3
def add_tax(df):
    df = df.copy()
    df["revenue_with_tax"] = df["revenue"] * 1.08
    return df

def test_add_tax():
    test_df = pd.DataFrame({"revenue": [100.0]})
    result = add_tax(test_df)
    expected = 108.0
    assert abs(result["revenue_with_tax"].iloc[0] - expected) < 0.01, "Tax calculation is wrong"
    print("test_add_tax: PASSED")

test_add_tax()


In [ ]:
# Solution — Exercise 4
def highest_revenue_store_BROKEN(df):
    totals = df.groupby("store")["revenue"].sum()
    return totals.sort_values().index[0]   # sort_values() defaults to ascending -- index[0] is the LOWEST, not highest

def highest_revenue_store_FIXED(df):
    totals = df.groupby("store")["revenue"].sum()
    return totals.sort_values(ascending=False).index[0]

print("Broken version returns (actually the LOWEST):", highest_revenue_store_BROKEN(sales))
print("Fixed version returns (the actual highest):", highest_revenue_store_FIXED(sales))


---
## Wrap-up

**Recap:** reading a traceback from the bottom up (error type + message, then the line number); common error types and what each usually signals (`KeyError`/`IndexError` for missing things, `TypeError`/`ValueError` for wrong types or values, `NameError` for undefined variables, `AttributeError` for a method that doesn't exist on that object); `assert` for turning a hidden assumption into something checked automatically; writing a simple test function with known input and a known correct output; and the harder case of a bug that doesn't crash, just silently gives a wrong answer.

**Common mistakes to watch for:**
- Reading a traceback top to bottom instead of starting from the last line
- Assuming code is correct just because it ran without crashing
- Using `and`/`or` instead of `&`/`|` on pandas boolean conditions
- Forgetting that `.sort_values()` defaults to ascending — `.index[0]` after a default sort gives you the *lowest* value, not the highest, unless you explicitly add `ascending=False`

**HW6 is due before this class.**

**No class Nov 25 — Thanksgiving.**

**Before next class (Nov 30):** Class 24 covers Git and reproducibility in more depth — version control as its own skill, not just a submission mechanism.
